In [27]:
#=========================================================================
#📘 FEATURE_EXTRACTION.PY (OPTIMIZADO PARA GITHUB)
#=========================================================================

#🎯 OBJETIVO:
#------------
#Cargar el dataset preprocesado (archivos .npz) y extraer embeddings 
#(vectores) por imagen usando MobileNetV2 preentrenada (sin la última 
#capa). Guardar los embeddings OPTIMIZADOS en embeddings.pkl

#✅ OPTIMIZACIONES IMPLEMENTADAS:
#---------------------------------
#1. Conversión a float16 (reduce 50% el tamaño)
#2. Reducción PCA opcional (reduce 80% más)
#3. Total: 50-85% de reducción según configuración

#📄 TECNOLOGÍA:
#-------------
#✅ MobileNetV2 (1280 dimensiones, 2018)
#✅ Compatible con TensorFlow 2.20+ y Keras 3.10+
#✅ Compatible con Python 3.13
#✅ Optimizado para GitHub (archivo <100MB)


In [28]:
# =========================================================================
# 📦 IMPORTAR LIBRERÍAS NECESARIAS
# =========================================================================

# =========================================================================
# 📘 FEATURE_EXTRACTION.PY - VERSIÓN CORREGIDA
# =========================================================================
# ✅ CORRECCIÓN: Mapeo robusto de labels a class_names

import os
import sys
import numpy as np
from tqdm import tqdm
import pickle
from pathlib import Path
import gc
import warnings

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.decomposition import PCA
from PIL import Image
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

np.random.seed(42)
tf.random.set_seed(42)

In [29]:
# =========================================================================
# ⚙️ CONFIGURACIÓN
# =========================================================================
class Config:
    DATASET_PATH = "../data/desayuno_preprocessed"
    OUTPUT_PATH = "../backend/models"
    OUTPUT_FILE = "embeddings.pkl"
    
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 32
    EMBEDDING_DIM = 1280
    
    USE_PCA = False
    PCA_COMPONENTS = 256
    USE_FLOAT16 = True
    GENERATE_PLOTS = True

os.makedirs(Config.OUTPUT_PATH, exist_ok=True)

In [30]:
# =========================================================================
# 🔧 FUNCIONES AUXILIARES
# =========================================================================
def load_npz_files(dataset_path):
    dataset_path = Path(dataset_path)
    npz_files = sorted(list(dataset_path.glob("*.npz")))
    
    if len(npz_files) == 0:
        raise FileNotFoundError(f"❌ No se encontraron archivos .npz en {dataset_path}")
    
    print(f"\n📦 Archivos .npz encontrados: {len(npz_files)}")
    print(f"📋 Primeros archivos:")
    for f in npz_files[:5]:
        print(f"   - {f.name}")
    if len(npz_files) > 5:
        print(f"   ... y {len(npz_files) - 5} más")
    
    return npz_files


def load_batch_from_npz(npz_file):
    data = np.load(npz_file)
    
    images = None
    for key in ['images', 'X', 'data', 'x']:
        if key in data:
            images = data[key]
            break
    
    if images is None and len(data.keys()) > 0:
        keys = list(data.keys())
        images = data[keys[0]]
    
    labels = None
    for key in ['labels', 'y', 'targets', 'label']:
        if key in data:
            labels = data[key]
            break
    
    if labels is None and len(data.keys()) > 1:
        keys = list(data.keys())
        labels = data[keys[1]]
    
    return images, labels


def preprocess_images_batch(images_batch):
    batch_processed = []
    
    for img_array in images_batch:
        try:
            if img_array.ndim == 2:
                img_array = np.expand_dims(img_array, axis=-1)
            
            if img_array.max() <= 1.0:
                img_array = (img_array * 255).astype(np.uint8)
            else:
                img_array = img_array.astype(np.uint8)
            
            if img_array.shape[-1] == 1:
                img_array = np.repeat(img_array, 3, axis=-1)
            
            if img_array.shape[-1] != 3:
                img_array = img_array[:, :, :3]
            
            if img_array.shape[:2] != Config.IMG_SIZE:
                img_pil = Image.fromarray(img_array)
                img_pil = img_pil.resize(Config.IMG_SIZE, Image.LANCZOS)
                img_array = np.array(img_pil, dtype=np.float32)
            else:
                img_array = img_array.astype(np.float32)
            
            batch_processed.append(img_array)
            
        except Exception as e:
            print(f"⚠️ Error procesando imagen: {e}")
            batch_processed.append(np.zeros((224, 224, 3), dtype=np.float32))
    
    batch_processed = np.array(batch_processed, dtype=np.float32)
    batch_processed = preprocess_input(batch_processed)
    
    return batch_processed


def extract_embeddings_from_npz_files(model, npz_files, batch_size):
    all_embeddings = []
    all_labels = []
    total_images_processed = 0
    
    print(f"\n🚀 Iniciando extracción de embeddings...")
    print(f"📦 Total archivos .npz: {len(npz_files)}")
    print(f"🔢 Batch size: {batch_size}\n")
    
    for npz_file in tqdm(npz_files, desc="Procesando archivos .npz"):
        images, labels = load_batch_from_npz(npz_file)
        
        if images is None:
            continue
        
        n_images = len(images)
        n_batches = (n_images + batch_size - 1) // batch_size
        
        for i in range(n_batches):
            start_idx = i * batch_size
            end_idx = min((i + 1) * batch_size, n_images)
            
            batch_images = images[start_idx:end_idx]
            batch_processed = preprocess_images_batch(batch_images)
            batch_embeddings = model.predict(batch_processed, verbose=0)
            
            all_embeddings.append(batch_embeddings)
            
            if labels is not None:
                batch_labels = labels[start_idx:end_idx]
                all_labels.extend(batch_labels)
            
            total_images_processed += len(batch_images)
            del batch_processed, batch_embeddings
        
        del images, labels
        gc.collect()
    
    embeddings = np.concatenate(all_embeddings, axis=0)
    labels_array = (
        np.array(all_labels) if all_labels 
        else np.zeros(len(embeddings), dtype=int)
    )
    
    print("\n✅ Extracción completada!")
    print(f"📊 Total imágenes procesadas: {total_images_processed:,}")
    print(f"📐 Shape final embeddings: {embeddings.shape}")
    print(f"📐 Shape final labels: {labels_array.shape}")
    
    return embeddings, labels_array


def apply_pca(embeddings, n_components=256):
    print("\n🔬 Aplicando PCA para reducir dimensionalidad...")
    print(f"📉 De {embeddings.shape[1]} → {n_components} dimensiones")
    
    pca = PCA(n_components=n_components)
    embeddings_pca = pca.fit_transform(embeddings)
    
    variance_explained = pca.explained_variance_ratio_.sum()
    
    print(f"✅ Dimensiones finales: {embeddings_pca.shape[1]}")
    print(f"📊 Varianza explicada: {variance_explained:.2%}")
    
    return embeddings_pca, pca


# ✅ FUNCIÓN CORREGIDA: Crear mapeo robusto label→class_name
def create_label_mapping(labels_array):
    """
    Crea un mapeo robusto entre índices de labels y nombres de clases
    
    Returns:
        tuple: (label_to_class_dict, class_names_list)
    """
    unique_labels = np.unique(labels_array)
    n_classes = len(unique_labels)
    
    print(f"\n🔍 Análisis de labels:")
    print(f"   Labels únicos: {n_classes}")
    print(f"   Rango: {unique_labels.min()} - {unique_labels.max()}")
    print(f"   Labels encontrados: {unique_labels[:10]}{'...' if len(unique_labels) > 10 else ''}")
    
    # Nombres predefinidos para 21 clases de desayuno
    predefined_names = [
        'pancakes', 'waffles', 'french_toast', 'eggs_benedict',
        'scrambled_eggs', 'omelette', 'fried_eggs', 'bacon',
        'sausage', 'hash_browns', 'toast', 'bagel',
        'croissant', 'muffin', 'cereal', 'oatmeal',
        'yogurt', 'fruit_salad', 'smoothie_bowl', 'avocado_toast',
        'breakfast_burrito'
    ]
    
    # ✅ CREAR MAPEO: label_value → class_name
    label_to_class = {}
    class_names = []
    
    for idx, label_value in enumerate(unique_labels):
        if n_classes == 21 and idx < len(predefined_names):
            class_name = predefined_names[idx]
        else:
            class_name = f"clase_{label_value}"
        
        label_to_class[label_value] = class_name
        class_names.append(class_name)
    
    print(f"\n✅ Mapeo creado:")
    print(f"   Ejemplo: label {unique_labels[0]} → '{class_names[0]}'")
    
    return label_to_class, class_names


def save_embeddings_optimized(embeddings, labels, label_to_class, class_names,
                               output_path, pca_model=None):
    """
    ✅ VERSIÓN CORREGIDA: Guarda con mapeo de labels
    """
    print("\n🔧 OPTIMIZANDO ARCHIVO PARA GITHUB")
    print("=" * 34)
    
    original_size_mb = embeddings.nbytes / (1024**2)
    print(f"\n📊 Tamaño original (float32): {original_size_mb:.2f} MB")
    
    if Config.USE_FLOAT16:
        embeddings_optimized = embeddings.astype(np.float16)
        optimized_size_mb = embeddings_optimized.nbytes / (1024**2)
        reduction_1 = (1 - optimized_size_mb / original_size_mb) * 100
        print(f"✅ Convertido a float16: {optimized_size_mb:.2f} MB "
              f"(reducción: {reduction_1:.1f}%)")
    else:
        embeddings_optimized = embeddings
    
    labels_optimized = labels.astype(np.int16)
    print(f"✅ Labels optimizados a int16")
    
    # ✅ GUARDAR CON MAPEO
    output_data = {
        'embeddings': embeddings_optimized,
        'labels': labels_optimized,
        'class_names': class_names,
        'label_to_class': label_to_class,  # ✅ NUEVO: Diccionario de mapeo
        'model': 'MobileNetV2',
        'embedding_dim': embeddings_optimized.shape[1],
        'dtype': 'float16' if Config.USE_FLOAT16 else 'float32',
        'pca_applied': Config.USE_PCA
    }
    
    if pca_model is not None:
        output_data['pca_model'] = pca_model
        print(f"✅ Modelo PCA incluido")
    
    print(f"\n💾 Guardando archivo embeddings.pkl...")
    
    with open(output_path, 'wb') as f:
        pickle.dump(output_data, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    final_size_mb = os.path.getsize(output_path) / (1024**2)
    total_reduction = (1 - final_size_mb / original_size_mb) * 100
    
    print("\n" + "="*70)
    print("✅ ARCHIVO GUARDADO EXITOSAMENTE")
    print("="*70)
    print(f"📊 Tamaño original (float32): {original_size_mb:.2f} MB")
    print(f"📦 Tamaño final (optimizado): {final_size_mb:.2f} MB")
    print(f"🎯 Reducción total: {total_reduction:.1f}%")
    print(f"📂 Ubicación: {output_path}")
    
    github_ok = "Sí ✅" if final_size_mb < 95 else "NO ❌"
    print(f"✅ Listo para GitHub (<100MB): {github_ok}")
    print("="*70)
    
    return final_size_mb


In [31]:
# =========================================================================
# 🚀 FUNCIÓN PRINCIPAL
# =========================================================================
def main():
    try:
        # PASO 1: Cargar archivos
        npz_files = load_npz_files(Config.DATASET_PATH)
        
        # PASO 2: Cargar modelo
        print("\n📄 Cargando MobileNetV2 preentrenado...")
        base_model = MobileNetV2(
            weights='imagenet',
            include_top=False,
            pooling='avg',
            input_shape=(224, 224, 3)
        )
        print("✅ Modelo cargado correctamente!")
        
        # PASO 3: Extraer embeddings
        embeddings, labels = extract_embeddings_from_npz_files(
            model=base_model,
            npz_files=npz_files,
            batch_size=Config.BATCH_SIZE
        )
        
        # PASO 4: ✅ CREAR MAPEO DE LABELS
        label_to_class, class_names = create_label_mapping(labels)
        
        # PASO 5: Aplicar PCA (opcional)
        pca_model = None
        if Config.USE_PCA:
            embeddings, pca_model = apply_pca(embeddings, Config.PCA_COMPONENTS)
        
        # PASO 6: ✅ GUARDAR CON MAPEO
        output_filepath = os.path.join(Config.OUTPUT_PATH, Config.OUTPUT_FILE)
        final_size = save_embeddings_optimized(
            embeddings=embeddings,
            labels=labels,
            label_to_class=label_to_class,  # ✅ NUEVO PARÁMETRO
            class_names=class_names,
            output_path=output_filepath,
            pca_model=pca_model
        )
        
        print("\n" + "="*70)
        print("✨ ¡Todo listo! Puedes continuar con el entrenamiento.")
        print("="*70)
        
        return 0
        
    except Exception as e:
        print(f"\n❌ ERROR: {str(e)}")
        import traceback
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    sys.exit(main())


📦 Archivos .npz encontrados: 1039
📋 Primeros archivos:
   - c000_batch0001.npz
   - c000_batch0002.npz
   - c000_batch0003.npz
   - c000_batch0004.npz
   - c000_batch0005.npz
   ... y 1034 más

📄 Cargando MobileNetV2 preentrenado...
✅ Modelo cargado correctamente!

🚀 Iniciando extracción de embeddings...
📦 Total archivos .npz: 1039
🔢 Batch size: 32



Procesando archivos .npz: 100%|██████████| 1039/1039 [15:23<00:00,  1.12it/s]



✅ Extracción completada!
📊 Total imágenes procesadas: 20,760
📐 Shape final embeddings: (20760, 1280)
📐 Shape final labels: (20760,)

🔍 Análisis de labels:
   Labels únicos: 21
   Rango: 0 - 20
   Labels encontrados: [0 1 2 3 4 5 6 7 8 9]...

✅ Mapeo creado:
   Ejemplo: label 0 → 'pancakes'

🔧 OPTIMIZANDO ARCHIVO PARA GITHUB

📊 Tamaño original (float32): 101.37 MB
✅ Convertido a float16: 50.68 MB (reducción: 50.0%)
✅ Labels optimizados a int16

💾 Guardando archivo embeddings.pkl...

✅ ARCHIVO GUARDADO EXITOSAMENTE
📊 Tamaño original (float32): 101.37 MB
📦 Tamaño final (optimizado): 50.72 MB
🎯 Reducción total: 50.0%
📂 Ubicación: ../backend/models\embeddings.pkl
✅ Listo para GitHub (<100MB): Sí ✅

✨ ¡Todo listo! Puedes continuar con el entrenamiento.


SystemExit: 0